# FloodLite — End-to-End Notebook (Kaggle / Colab / M2 Pro)

This notebook reproduces every numerical result in the FloodLite manuscript.
It is designed to run on:
- **Kaggle** (free Tesla T4, 30 hr/week) — recommended
- **Google Colab** (free T4 or A100 in Pro)
- **MacBook M2 Pro** (CPU + MPS) — slower, but works for inference and small fine-tunes

## What it does
1. Mounts the public FSSD dataset at the Kaggle layout (`dataset/{train,val}/{images,labels}`)
2. Trains a UNet + EfficientNet-B0 **teacher** (5-fold CV by default)
3. Trains three **students** (MobileNetV3-Small, EfficientNet-Lite0, MobileViT-XXS) under four KD configs (no-KD, response-only, feature-only, combined)
4. Applies **post-training INT8 quantization**
5. **Benchmarks latency** on the host CPU/MPS
6. Exports an ONNX file for downstream CoreML / TFLite / WASM conversion
7. Prints a **results table ready to paste into the manuscript**

Total wall-time: ~3–4 hours on Kaggle T4 for a single fold; ~1 day for full 5-fold.

## 1. Install dependencies

In [ ]:
%pip install -q segmentation_models_pytorch==0.3.4 timm==1.0.11 albumentations==1.4.18 opencv-python-headless torchprofile tqdm onnx onnxruntime

## 2. Imports & device selection

In [ ]:
import os, sys, json, time, copy, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# Pick device: CUDA > MPS (Apple) > CPU
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

## 3. FSSD dataset

On Kaggle: add the dataset *Flood Semantic Segmentation Dataset* (lihuayang111265) via the right-side panel. It mounts at `/kaggle/input/flood-semantic-segmentation-dataset/`.

**Layout in this Kaggle dataset:**
```
/kaggle/input/flood-semantic-segmentation-dataset/
└── dataset/
    ├── train/
    │   ├── images/  (RGB)
    │   └── labels/  (binary masks)
    └── val/
        ├── images/
        └── labels/
```

By default we **combine train + val into one set and run 5-fold CV on it** — this matches the methodology of Karcı et al. (2026) and gives more robust statistics on a small dataset (~663 images). Set `USE_PREDEFINED_SPLIT = True` below if you'd rather use the dataset's built-in train/val split.

Locally / Colab: download from <https://www.kaggle.com/datasets/lihuayang111265/flood-semantic-segmentation-dataset> and unzip.

In [ ]:
# EDIT THIS PATH IF NEEDED
DATA_ROOT = Path('/kaggle/input/flood-semantic-segmentation-dataset')
USE_PREDEFINED_SPLIT = False  # True = use train/ for training and val/ for validation; False = combine and 5-fold CV

IMG_EXTS  = {'.jpg', '.jpeg', '.png'}
MASK_EXTS = ['.png', '.jpg', '.jpeg']  # tried in order when finding mask for a given image stem

def find_split_dirs(root: Path, split: str):
    """Locate (images_dir, labels_dir) for split in {'train','val'}, robust to layout variations."""
    # Try several common layouts
    candidates = list(root.rglob(f'{split}/images')) + list(root.rglob(f'{split}/Image')) + \
                 list(root.rglob(f'{split}/IMAGES'))
    if not candidates:
        return None, None
    img_dir = candidates[0]
    # labels folder lives next to images
    for lbl_name in ('labels', 'Mask', 'masks', 'Labels', 'mask'):
        lbl_dir = img_dir.parent / lbl_name
        if lbl_dir.exists():
            return img_dir, lbl_dir
    return img_dir, None

def gather_pairs(img_dir: Path, lbl_dir: Path):
    pairs = []
    if img_dir is None or lbl_dir is None:
        return pairs
    for ip in sorted(img_dir.iterdir()):
        if ip.suffix.lower() not in IMG_EXTS:
            continue
        # find mask with same stem
        for ext in MASK_EXTS:
            mp = lbl_dir / (ip.stem + ext)
            if mp.exists():
                pairs.append((ip, mp))
                break
    return pairs

train_img_dir, train_lbl_dir = find_split_dirs(DATA_ROOT, 'train')
val_img_dir,   val_lbl_dir   = find_split_dirs(DATA_ROOT, 'val')
TRAIN_PAIRS = gather_pairs(train_img_dir, train_lbl_dir)
VAL_PAIRS   = gather_pairs(val_img_dir,   val_lbl_dir)
ALL_PAIRS   = TRAIN_PAIRS + VAL_PAIRS

print(f'Detected train images dir: {train_img_dir}')
print(f'Detected train labels dir: {train_lbl_dir}')
print(f'Detected val   images dir: {val_img_dir}')
print(f'Detected val   labels dir: {val_lbl_dir}')
print(f'Pairs found — train: {len(TRAIN_PAIRS)}   val: {len(VAL_PAIRS)}   total: {len(ALL_PAIRS)}')
assert len(ALL_PAIRS) > 0, 'No image/mask pairs found — check DATA_ROOT and folder structure.'

In [ ]:
IMG_SIZE = 256

def get_transforms(train: bool):
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class FSSD(Dataset):
    """FSSD dataset taking a list of (image_path, mask_path) pairs directly."""
    def __init__(self, pairs, indices=None, train=True):
        self.pairs = [pairs[i] for i in indices] if indices is not None else list(pairs)
        self.tfm = get_transforms(train)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        m   = (cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        out = self.tfm(image=img, mask=m)
        return out['image'], out['mask'].unsqueeze(0)

# Build the splits
if USE_PREDEFINED_SPLIT:
    # one-shot mode: train on TRAIN_PAIRS, validate on VAL_PAIRS
    FOLDS = [(np.arange(len(TRAIN_PAIRS)), np.arange(len(VAL_PAIRS)) + len(TRAIN_PAIRS))]
    PAIRS = ALL_PAIRS  # FOLDS index into this combined list
    print(f'Using predefined split: {len(TRAIN_PAIRS)} train, {len(VAL_PAIRS)} val (single fold)')
else:
    # Combine and 5-fold CV (matches Karcı et al. 2026 methodology)
    PAIRS = ALL_PAIRS
    n_total = len(PAIRS)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    FOLDS = list(kf.split(np.arange(n_total)))
    print(f'5-fold CV ready over {n_total} combined images')

## 4. Models — Teacher and three students

Teacher: UNet + EfficientNet-B0  
Students: UNet + {MobileNetV3-Small, EfficientNet-Lite0, MobileViT-XXS}

In [ ]:
STUDENT_BACKBONES = {
    'mobilenetv3_small': 'tu-mobilenetv3_small_100',
    'efficientnet_lite0': 'timm-tf_efficientnet_lite0',
    'mobilevit_xxs':      'tu-mobilevit_xxs',
}

def make_teacher():
    return smp.Unet(encoder_name='efficientnet-b0', encoder_weights='imagenet',
                    in_channels=3, classes=1, activation=None)

def make_student(name):
    return smp.Unet(encoder_name=STUDENT_BACKBONES[name], encoder_weights='imagenet',
                    in_channels=3, classes=1, activation=None)

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

def estimate_gflops(m, sz=IMG_SIZE):
    try:
        from torchprofile import profile_macs
        x = torch.randn(1, 3, sz, sz)
        return profile_macs(m.cpu().eval(), x) * 2 / 1e9
    except Exception:
        return float('nan')

for name in ['teacher (B0)'] + list(STUDENT_BACKBONES):
    m = make_teacher() if name.startswith('teacher') else make_student(name)
    print(f'{name:30s}  params={count_params(m):.2f} M   FLOPs={estimate_gflops(m):.2f} G')
    del m

## 5. Losses (BCE + Dice + KD)

In [ ]:
def dice_loss(logits, target, smooth=1.0):
    p = torch.sigmoid(logits).flatten(1)
    t = target.flatten(1)
    inter = (p*t).sum(1)
    return (1 - (2*inter + smooth) / (p.sum(1) + t.sum(1) + smooth)).mean()

def task_loss(logits, target):
    return 0.5*F.binary_cross_entropy_with_logits(logits, target) + 0.5*dice_loss(logits, target)

def response_kd_loss(s_logits, t_logits, T=4.0):
    p_t = torch.sigmoid(t_logits / T).detach()
    log_q  = F.logsigmoid(s_logits / T)
    log_qn = F.logsigmoid(-s_logits / T)
    return -(p_t*log_q + (1-p_t)*log_qn).mean() * (T*T)

class FeatureKDLoss(nn.Module):
    def __init__(self, c_s, c_t):
        super().__init__()
        self.adapter = nn.Conv2d(c_s, c_t, 1, bias=False)
    def forward(self, fs, ft):
        if fs.shape[-2:] != ft.shape[-2:]:
            fs = F.interpolate(fs, size=ft.shape[-2:], mode='bilinear', align_corners=False)
        return F.mse_loss(self.adapter(fs), ft.detach())

## 6. Metrics

In [ ]:
@torch.no_grad()
def metrics(logits, target, thr=0.5):
    pred = (torch.sigmoid(logits) >= thr).float()
    tgt  = (target >= 0.5).float()
    tp = (pred*tgt).sum(); fp = (pred*(1-tgt)).sum(); fn = ((1-pred)*tgt).sum(); tn = ((1-pred)*(1-tgt)).sum()
    eps = 1e-8
    acc = (tp+tn) / (tp+tn+fp+fn+eps)
    pre = tp / (tp+fp+eps)
    rec = tp / (tp+fn+eps)
    f1  = 2*pre*rec / (pre+rec+eps)
    iou = tp / (tp+fp+fn+eps)
    return dict(accuracy=acc.item(), precision=pre.item(), recall=rec.item(), f1=f1.item(), iou=iou.item())

def aggregate(loaders, model):
    model.eval()
    keys = ['accuracy','precision','recall','f1','iou']
    sums = {k: 0.0 for k in keys}; n = 0
    with torch.no_grad():
        for x, y in loaders:
            x, y = x.to(DEVICE), y.to(DEVICE)
            m = metrics(model(x), y)
            for k in keys:
                sums[k] += m[k]
            n += 1
    return {k: sums[k]/max(n,1) for k in keys}

## 7. Train teacher (single fold demo; loop over folds for full CV)

In [ ]:
FOLD = 0          # ← change to 0..4 for full 5-fold CV (when USE_PREDEFINED_SPLIT=False)
EPOCHS_T = 50
BATCH = 8 if DEVICE == 'cuda' else 4

tr_idx, va_idx = FOLDS[FOLD]
tr_loader = DataLoader(FSSD(PAIRS, tr_idx.tolist(), train=True),  batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
va_loader = DataLoader(FSSD(PAIRS, va_idx.tolist(), train=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Fold {FOLD}: {len(tr_loader.dataset)} train, {len(va_loader.dataset)} val')

teacher = make_teacher().to(DEVICE)
opt = AdamW(teacher.parameters(), lr=1e-4, weight_decay=1e-4)
sch = CosineAnnealingLR(opt, T_max=EPOCHS_T)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE=='cuda')

best_iou = 0.0
Path('ckpts').mkdir(exist_ok=True)
for ep in range(EPOCHS_T):
    teacher.train()
    for x, y in tr_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=DEVICE=='cuda'):
            loss = task_loss(teacher(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sch.step()
    m = aggregate(va_loader, teacher)
    print(f'[teacher] ep {ep+1:02d}  val_IoU={m["iou"]:.4f}  F1={m["f1"]:.4f}')
    if m['iou'] > best_iou:
        best_iou = m['iou']
        torch.save(teacher.state_dict(), f'ckpts/teacher_fold{FOLD}.pt')
print('Best teacher IoU:', best_iou)

## 8. Train students (4 KD configs × 3 students)

**KD configs**: `none` (task only), `resp` (response only), `feat` (feature only), `comb` (response + feature).

In [ ]:
EPOCHS_S = 50
ALPHA = 0.5; BETA = 0.1; T_KD = 4.0

# Reload best teacher
teacher = make_teacher().to(DEVICE).eval()
teacher.load_state_dict(torch.load(f'ckpts/teacher_fold{FOLD}.pt', map_location=DEVICE))
for p in teacher.parameters(): p.requires_grad = False

class FeatHook:
    def __init__(self, mod): self.f = None; self.h = mod.register_forward_hook(self._h)
    def _h(self, m, i, o): self.f = o
    def close(self): self.h.remove()

results = {}
for sname in ['mobilenetv3_small', 'efficientnet_lite0', 'mobilevit_xxs']:
    for kd in ['none', 'resp', 'feat', 'comb']:
        student = make_student(sname).to(DEVICE)
        opt_params = list(student.parameters())
        s_h = t_h = feat_loss = None
        if kd in ('feat', 'comb'):
            s_h = FeatHook(student.decoder.blocks[2]); t_h = FeatHook(teacher.decoder.blocks[2])
            with torch.no_grad():
                xp, _ = next(iter(tr_loader)); xp = xp[:1].to(DEVICE)
                _ = student(xp); _ = teacher(xp)
                feat_loss = FeatureKDLoss(s_h.f.shape[1], t_h.f.shape[1]).to(DEVICE)
                opt_params += list(feat_loss.parameters())
        opt = AdamW(opt_params, lr=5e-4, weight_decay=1e-4)
        sch = CosineAnnealingLR(opt, T_max=EPOCHS_S)
        scaler = torch.cuda.amp.GradScaler(enabled=DEVICE=='cuda')
        best = 0.0
        for ep in range(EPOCHS_S):
            student.train()
            for x, y in tr_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                with torch.cuda.amp.autocast(enabled=DEVICE=='cuda'):
                    with torch.no_grad():
                        t_logits = teacher(x)
                    s_logits = student(x)
                    L = task_loss(s_logits, y)
                    if kd in ('resp','comb'):
                        L = (1-ALPHA)*L + ALPHA*response_kd_loss(s_logits, t_logits, T=T_KD)
                    if kd in ('feat','comb'):
                        L = L + BETA*feat_loss(s_h.f, t_h.f)
                scaler.scale(L).backward(); scaler.step(opt); scaler.update()
            sch.step()
            m = aggregate(va_loader, student)
            if m['iou'] > best:
                best = m['iou']
                torch.save(student.state_dict(), f'ckpts/{sname}_{kd}_fold{FOLD}.pt')
        if s_h: s_h.close(); t_h.close()
        m = aggregate(va_loader, student)
        results[(sname, kd)] = m
        print(f'[{sname:22s} kd={kd:4s}] IoU={m["iou"]:.4f}  F1={m["f1"]:.4f}')

print('\nFold', FOLD, 'student grid done.')

## 9. INT8 Quantization (CPU only — PyTorch quantization is x86)

In [ ]:
def quantize_int8(model, calib_loader, n_calib=25):
    m = copy.deepcopy(model).cpu().eval()
    m.qconfig = torch.ao.quantization.get_default_qconfig('x86')
    torch.ao.quantization.prepare(m, inplace=True)
    with torch.no_grad():
        for i, (x, _) in enumerate(calib_loader):
            m(x)
            if i+1 >= n_calib: break
    torch.ao.quantization.convert(m, inplace=True)
    return m

def model_size_mb(model):
    import io; b = io.BytesIO(); torch.save(model.state_dict(), b); return b.tell()/(1024**2)

quant_results = {}
for sname in ['mobilenetv3_small', 'efficientnet_lite0', 'mobilevit_xxs']:
    student_fp = make_student(sname)
    student_fp.load_state_dict(torch.load(f'ckpts/{sname}_comb_fold{FOLD}.pt', map_location='cpu'))
    student_fp = student_fp.eval()
    fp_size = model_size_mb(student_fp)
    fp_metrics = aggregate(va_loader, student_fp.to(DEVICE))
    try:
        student_q = quantize_int8(student_fp, tr_loader)
        q_size = model_size_mb(student_q)
        q_metrics = aggregate(va_loader, student_q.cpu())
        quant_results[sname] = {'fp_iou': fp_metrics['iou'], 'q_iou': q_metrics['iou'],
                                'fp_size_mb': fp_size, 'q_size_mb': q_size}
        print(f'[{sname:22s}] FP32 IoU={fp_metrics["iou"]:.4f}  INT8 IoU={q_metrics["iou"]:.4f}  size {fp_size:.2f}→{q_size:.2f} MB')
    except Exception as e:
        print(f'[{sname}] quantization failed: {e}; will fall back to FP32 in deployment.')

## 10. Latency benchmark (host CPU/MPS/GPU)

In [ ]:
@torch.no_grad()
def latency(model, device, n_warm=20, n_iter=200):
    model = model.to(device).eval()
    x = torch.randn(1,3,IMG_SIZE,IMG_SIZE, device=device)
    for _ in range(n_warm): _ = model(x)
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(n_iter):
        t0 = time.perf_counter(); _ = model(x)
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter()-t0)*1000)
    arr = np.asarray(times)
    return dict(p50=float(np.percentile(arr,50)), p95=float(np.percentile(arr,95)), fps=1000/np.percentile(arr,50))

print(f'Latency on host device {DEVICE} (256×256 input):')
lat_teacher = latency(make_teacher().to(DEVICE), DEVICE)
print(f'  Teacher (B0):                p50={lat_teacher["p50"]:.2f} ms  p95={lat_teacher["p95"]:.2f} ms  fps={lat_teacher["fps"]:.1f}')
for sname in ['mobilenetv3_small', 'efficientnet_lite0', 'mobilevit_xxs']:
    s = make_student(sname); s.load_state_dict(torch.load(f'ckpts/{sname}_comb_fold{FOLD}.pt', map_location=DEVICE))
    l = latency(s, DEVICE)
    print(f'  {sname:22s} (FP32):       p50={l["p50"]:.2f} ms  p95={l["p95"]:.2f} ms  fps={l["fps"]:.1f}')

## 11. Export to ONNX (for CoreML / TFLite / WASM downstream)

In [ ]:
Path('exports').mkdir(exist_ok=True)
for sname in ['mobilenetv3_small', 'efficientnet_lite0', 'mobilevit_xxs']:
    s = make_student(sname).cpu().eval()
    s.load_state_dict(torch.load(f'ckpts/{sname}_comb_fold{FOLD}.pt', map_location='cpu'))
    out = f'exports/{sname}.onnx'
    torch.onnx.export(s, torch.randn(1,3,IMG_SIZE,IMG_SIZE), out, opset_version=13,
                      input_names=['input'], output_names=['logits'], dynamo=False)
    print('Exported', out)

## 12. Print a results table ready to paste into the manuscript

Copy the printed values into Tables 2–6 of `FloodLite_Manuscript.docx`.

In [ ]:
print('=== Table 3: Students (no KD vs FloodLite KD) ===')
print(f'{"Configuration":40s} {"IoU":>8s} {"F1":>8s} {"Δ vs T":>8s}')
t_iou = best_iou
for sname in ['mobilenetv3_small', 'efficientnet_lite0', 'mobilevit_xxs']:
    no = results.get((sname, 'none'), {}).get('iou', float('nan'))
    cb = results.get((sname, 'comb'), {}).get('iou', float('nan'))
    print(f'{sname:22s}  no-KD       {no:8.4f} {results.get((sname,"none"),{}).get("f1",0):8.4f} {no-t_iou:+8.4f}')
    print(f'{sname:22s}  FloodLite KD{cb:8.4f} {results.get((sname,"comb"),{}).get("f1",0):8.4f} {cb-t_iou:+8.4f}')

print('\n=== Table 4: KD ablation on MobileViT-XXS-UNet ===')
for kd in ['none','resp','feat','comb']:
    iou = results.get(('mobilevit_xxs', kd), {}).get('iou', float('nan'))
    print(f'  {kd:6s}  IoU={iou:.4f}')

print('\n=== Table 5: Quantization ===')
for k, v in quant_results.items():
    print(f'  {k}  FP32 IoU={v["fp_iou"]:.4f}  INT8 IoU={v["q_iou"]:.4f}  Δ={v["q_iou"]-v["fp_iou"]:+.4f}  size {v["fp_size_mb"]:.2f}→{v["q_size_mb"]:.2f} MB')